In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr

# CMIP6

In [2]:
inst = {
    "ACCESS-CM2": "CSIRO-ARCCSS", "ACCESS-ESM1-5": "CSIRO", "AWI-CM-1-1-MR": "AWI", "BCC-CSM2-MR": "BCC", "CAMS-CSM1-0": "CAMS",
    "CanESM5": "CCCma", "CESM2": "NCAR", "CESM2-WACCM": "NCAR", "CMCC-CM2-SR5": "CMCC", "CNRM-CM6-1": "CNRM-CERFACS", "CNRM-CM6-1-HR": "CNRM-CERFACS",
    "CNRM-ESM2-1": "CNRM-CERFACS", "EC-Earth3": "EC-Earth-Consortium", "EC-Earth3-Veg": "EC-Earth-Consortium", "FGOALS-g3": "CAS",
    "GFDL-CM4": "NOAA-GFDL", "GFDL-ESM4": "NOAA-GFDL", "HadGEM3-GC31-LL": "MOHC", "IITM-ESM": "CCCR-IITM", "INM-CM4-8": "INM", "INM-CM5-0": "INM",
    "IPSL-CM6A-LR": "IPSL", "KACE-1-0-G": "NIMS-KMA", "KIOST-ESM": "KIOST", "MIROC6": "MIROC", "MIROC-ES2L": "MIROC", "MPI-ESM-1-2-HAM": "HAMMOZ-Consortium",
    "MPI-ESM1-2-HR": "MPI-M", "MPI-ESM1-2-LR": "MPI-M", "MRI-ESM2-0": "MRI", "NESM3": "NUIST", "NorESM2-LM": "NCC", "NorESM2-MM": "NCC",
    "TaiESM1": "AS-RCEC", "UKESM1-0-LL": "MOHC", }

grid = {
    "ACCESS-CM2": "gn", "ACCESS-ESM1-5": "gn", "AWI-CM-1-1-MR": "gn", "BCC-CSM2-MR": "gn", "CAMS-CSM1-0": "gn", "CanESM5": "gn",
    "CESM2": "gn", "CESM2-WACCM": "gn", "CMCC-CM2-SR5": "gn", "CNRM-CM6-1": "gr", "CNRM-CM6-1-HR": "gr", "CNRM-ESM2-1": "gr",
    "EC-Earth3": "gr", "EC-Earth3-Veg": "gr", "FGOALS-g3": "gn", "GFDL-CM4": "gr1", "GFDL-ESM4": "gr1", "HadGEM3-GC31-LL": "gn",
    "IITM-ESM": "gn", "INM-CM4-8": "gr1", "INM-CM5-0": "gr1", "IPSL-CM6A-LR": "gr", "KACE-1-0-G": "gr", "KIOST-ESM": "gr1",
    "MIROC-ES2L": "gn", "MIROC6": "gn", "MPI-ESM-1-2-HAM": "gn", "MPI-ESM1-2-HR": "gn", "MPI-ESM1-2-LR": "gn", "MRI-ESM2-0": "gn",
    "NESM3": "gn", "NorESM2-LM": "gn", "NorESM2-MM": "gn", "TaiESM1": "gn", "UKESM1-0-LL": "gn", }

In [3]:
atlas_original = pd.read_csv("atlas/CMIP6_day.csv")

df = atlas_original["Simulation"].str.split("_", expand=True).rename(
    {0: "project", 1: "source_id", 2: "experiment_id", 3: "variant_label"},
    axis=1)

df["version"] = atlas_original["tas"].str.split(",", expand=True)[0]
df["institute_id"] = df["source_id"].map(inst)
df["grid_label"] = df["source_id"].map(grid)
df["table_id"] = "day"
df["variable_id"] = "tas"
df["activity_id"] = df["experiment_id"].map(lambda x: "CMIP" if x == "historical" else "ScenarioMIP")

df = df[(df["version"] != "FALSE")]
df

,project,source_id,experiment_id,variant_label,version,institute_id,grid_label,table_id,variable_id,activity_id
0,CMIP6,ACCESS-CM2,historical,r1i1p1f1,v20191108,CSIRO-ARCCSS,gn,day,tas,CMIP
1,CMIP6,ACCESS-CM2,ssp126,r1i1p1f1,v20191108,CSIRO-ARCCSS,gn,day,tas,ScenarioMIP
2,CMIP6,ACCESS-CM2,ssp245,r1i1p1f1,v20191108,CSIRO-ARCCSS,gn,day,tas,ScenarioMIP
3,CMIP6,ACCESS-CM2,ssp370,r1i1p1f1,v20191108,CSIRO-ARCCSS,gn,day,tas,ScenarioMIP
4,CMIP6,ACCESS-CM2,ssp585,r1i1p1f1,v20191108,CSIRO-ARCCSS,gn,day,tas,ScenarioMIP
...,...,...,...,...,...,...,...,...,...,...
164,CMIP6,UKESM1-0-LL,historical,r1i1p1f2,v20190627,MOHC,gn,day,tas,CMIP
165,CMIP6,UKESM1-0-LL,ssp126,r1i1p1f2,v20190708,MOHC,gn,day,tas,ScenarioMIP
166,CMIP6,UKESM1-0-LL,ssp245,r1i1p1f2,v20190715,MOHC,gn,day,tas,ScenarioMIP
167,CMIP6,UKESM1-0-LL,ssp370,r1i1p1f2,v20190726,MOHC,gn,day,tas,ScenarioMIP


In [4]:
dataset_facets = ["project", "activity_id", "institute_id", "source_id", "experiment_id", "variant_label", "table_id", "variable_id", "grid_label", "version"]
instance_ids   = df[dataset_facets].astype(str).agg(".".join, axis=1)

In [5]:
instance_ids

0      CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....
1      CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp1...
2      CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp2...
3      CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp3...
4      CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp5...
                             ...                        
164    CMIP6.CMIP.MOHC.UKESM1-0-LL.historical.r1i1p1f...
165    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp126.r1i1...
166    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245.r1i1...
167    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp370.r1i1...
168    CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1...
Length: 167, dtype: object

In [6]:
instance_ids.str.split(".", expand=True).rename(dict(zip(range(len(dataset_facets)), dataset_facets)), axis=1).join(
    instance_ids.rename("instance_id")).to_csv("atlas.csv", index=False, header=True)

Now I have the ESGF inventory.

In [7]:
esgf = pd.read_json("esgf.json", lines=True)
esgf

,id,version,access,activity_drs,activity_id,cf_standard_name,citation_url,data_node,data_specs_version,dataset_id_template_,...,variant_label,west_degrees,xlink,retracted,_timestamp,score,_version_,number_of_aggregations,url,branch_method
0,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,20191108,HTTPServer,CMIP,CMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f1,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-01-23T14:35:08.584Z,1,1803475775129649152,NaN,NaN,NaN
1,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,20191108,HTTPServer,CMIP,CMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.30,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f1,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-10-29T12:49:43.606Z,1,1681890511270445056,2.0,[http://esgf3.dkrz.de/thredds/catalog/esgcet/1...,NaN
2,CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp1...,20191108,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f1,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2021-06-17T10:58:39.601Z,1,1803477063236059136,NaN,NaN,NaN
3,CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp1...,20191108,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.30,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f1,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-11-03T16:05:59.825Z,1,1700915415751852032,2.0,[http://esgf3.dkrz.de/thredds/catalog/esgcet/1...,NaN
4,CMIP6.ScenarioMIP.CSIRO-ARCCSS.ACCESS-CM2.ssp2...,20191108,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f1,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-01-21T09:26:25.967Z,1,1803477063760347136,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp126.r1i1...,20190708,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.29,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f2,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2019-07-08T13:30:11.308Z,1,1803477129903472640,NaN,NaN,NaN
307,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp126.r1i1...,20190708,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f2,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-11-07T07:03:16.714Z,1,1682684087333027840,2.0,[http://esgf3.dkrz.de/thredds/catalog/esgcet/1...,NaN
308,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245.r1i1...,20190715,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f2,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-11-07T07:59:16.531Z,1,1682687610355449856,2.0,[http://esgf3.dkrz.de/thredds/catalog/esgcet/3...,NaN
309,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp370.r1i1...,20190726,HTTPServer,ScenarioMIP,ScenarioMIP,air_temperature,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,%(mip_era)s.%(activity_drs)s.%(institution_id)...,...,r1i1p1f2,0.9375,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,False,2020-11-07T08:42:51.325Z,1,1682690352165683200,2.0,[http://esgf3.dkrz.de/thredds/catalog/esgcet/3...,NaN


In [8]:
len(instance_ids), len(esgf["instance_id"].unique())

(167, 161)

In [9]:
[x for x in instance_ids.values if x not in esgf["instance_id"].unique() or x in esgf[esgf["retracted"]]["instance_id"].values]

['CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126',
 'CMIP6.CMIP.NCAR.CESM2.historical.r4i1p1f1.day.tas.gn.v20190401',
 'CMIP6.CMIP.EC-Earth-Consortium.EC-Earth3-Veg.historical.r1i1p1f1.day.tas.gr.v20200225',
 'CMIP6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-Veg.ssp126.r1i1p1f1.day.tas.gr.v20200225',
 'CMIP6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-Veg.ssp245.r1i1p1f1.day.tas.gr.v20200225',
 'CMIP6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-Veg.ssp370.r1i1p1f1.day.tas.gr.v20200225',
 'CMIP6.ScenarioMIP.KIOST.KIOST-ESM.ssp245.r1i1p1f1.day.tas.gr1.v20191202',
 'CMIP6.ScenarioMIP.MIROC.MIROC-ES2L.ssp126.r1i1p1f2.day.tas.gn.v20200318',
 'CMIP6.ScenarioMIP.MIROC.MIROC-ES2L.ssp585.r1i1p1f2.day.tas.gn.v20200318',
 'CMIP6.ScenarioMIP.MPI-M.MPI-ESM1-2-HR.ssp126.r1i1p1f1.day.tas.gn.v20190710',
 'CMIP6.ScenarioMIP.MPI-M.MPI-ESM1-2-HR.ssp245.r1i1p1f1.day.tas.gn.v20190710',
 'CMIP6.ScenarioMIP.MPI-M.MPI-ESM1-2-HR.ssp370.r1i1p1f1.day.tas.gn.v20190710',
 'CMIP6.ScenarioMIP.MPI-M.M

Ok, esta info es de acuerdo al index, estoy descargando los datos que supuestamente están disponibles a ver que ficheros faltan etc. Puede ser interesante ver que ficheros están disponibles en cada data node, por ver que data nodes tienen malas replicas. Se pueden ver más cosas:

- Checksums contradictorios entre réplicas o incorrectos, etc
- Timestamps de ficheros

Para todo esto hace falta inventario de ficheros (además del de datasets).

In [10]:
esgf_files = pd.read_json("esgf-files.json", lines=True)
esgf_files

,id,version,activity_drs,activity_id,cf_standard_name,checksum,checksum_type,citation_url,data_node,data_specs_version,...,variable,variable_id,variable_long_name,variable_units,variant_label,retracted,_timestamp,score,_version_,branch_method
0,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,1,CMIP,CMIP,air_temperature,7cb26ece5ba2274e8878eea0a32608d751caeebdc78845...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f1,False,2020-01-23T14:35:09.111Z,1,1803414634953179137,NaN
1,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,1,CMIP,CMIP,air_temperature,20e949eaaaf640495f56dced938499f273b8fa0ffa10f6...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f1,False,2020-01-23T14:35:08.951Z,1,1803414634954227712,NaN
2,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,1,CMIP,CMIP,air_temperature,b469e5a706009475f72c84a127a52d8cfdd22b265f4704...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f1,False,2020-01-23T14:35:08.881Z,1,1803414634954227713,NaN
3,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,1,CMIP,CMIP,air_temperature,f17b11cf4f55d746799a64073359741397ba30461eae33...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.ceda.ac.uk,01.00.30,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f1,False,2020-01-23T14:35:08.750Z,1,1803414634955276288,NaN
4,CMIP6.CMIP.CSIRO-ARCCSS.ACCESS-CM2.historical....,1,CMIP,CMIP,air_temperature,7cb26ece5ba2274e8878eea0a32608d751caeebdc78845...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.30,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f1,False,2020-10-29T12:49:43.618Z,1,1681890511284076544,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5209,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245.r1i1...,1,ScenarioMIP,ScenarioMIP,air_temperature,bf0defb2c5619aa151727144da8792bba20fb20d45218a...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f2,False,2020-11-07T07:59:16.537Z,1,1682687610361741312,NaN
5210,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp370.r1i1...,1,ScenarioMIP,ScenarioMIP,air_temperature,95f183c3be024c901726a5ca9ded4f5c8fd2d69eba8b85...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f2,False,2020-11-07T08:42:51.335Z,1,1682690352176168960,NaN
5211,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp370.r1i1...,1,ScenarioMIP,ScenarioMIP,air_temperature,12c13dbb902022cc804c6c9c53d36a1ed2a38217e65d51...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f2,False,2020-11-07T08:42:51.332Z,1,1682690352173023232,NaN
5212,CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1...,1,ScenarioMIP,ScenarioMIP,air_temperature,dc32687dadfce2df96e40cad1c11c6c1eab7185313bdbb...,SHA256,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf3.dkrz.de,01.00.29,...,tas,tas,Near-Surface Air Temperature,K,r1i1p1f2,False,2020-11-07T17:16:59.800Z,1,1682722699137253376,NaN


In [11]:
esgf_files[["data_node", "title"]].groupby("data_node").count().join(
    esgf_files[["data_node", "replica"]].groupby("data_node").sum())

,title,replica
data_node,,
esg-cccr.tropmet.res.in,101,0
esg.camscma.cn,5,0
esg1.umr-cnrm.fr,23,0
esgdata.gfdl.noaa.gov,48,0
esgf-node2.cmcc.it,23,0
esgf.ceda.ac.uk,1972,1962
esgf.rcec.sinica.edu.tw,26,0
esgf3.dkrz.de,3008,2401
vesg.ipsl.upmc.fr,8,0


In [12]:
esgf_files["dataset_id"].str.split("|").str[0].unique().size

160

In [13]:
# cual es el q falta para q sea 161
[x for x in esgf["instance_id"].unique() if x not in esgf_files["dataset_id"].str.split("|").str[0].unique()]

['CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126']

In [14]:
esgf[esgf["instance_id"] == "CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126"].iloc[0]["id"]

'CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126|esgf.ceda.ac.uk'

```bash
$ esgfsearch.py -q "project=CMIP6 dataset_id=CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126|esgf.ceda.ac.uk type=File"
 
Query results for https://esgf-ui.ceda.ac.uk/proxy/search?project=CMIP6&dataset_id=CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126%7Cesgf.ceda.ac.uk&type=File&limit=0&format=application%2Fsolr%2Bjson
Found 0 for https://esgf-ui.ceda.ac.uk/proxy/search?project=CMIP6&dataset_id=CMIP6.CMIP.BCC.BCC-CSM2-MR.historical.r1i1p1f1.day.tas.gn.v20181126%7Cesgf.ceda.ac.uk&type=File&limit=0&format=application%2Fsolr%2Bjson
```

In [15]:
esgf_files["checksum"].unique().size, esgf_files["title"].unique().size

(3044, 3044)

What about timestamps?

In [16]:
tss = pd.to_datetime(esgf["_timestamp"].str.split(".").str[0].str.replace("Z",""))
vs = pd.to_datetime(esgf["version"], format="%Y%m%d")

(tss - vs).sort_values()

37       0 days 02:59:22
301      0 days 06:00:06
303      0 days 11:20:03
306      0 days 13:30:11
42       0 days 23:29:48
             ...        
240   1181 days 09:51:41
144   1203 days 14:43:41
142   1203 days 14:57:29
146   1203 days 14:57:32
297   1333 days 12:30:09
Length: 311, dtype: timedelta64[ns]

In [17]:
tss_files = pd.to_datetime(esgf_files["_timestamp"].str.split(".").str[0].str.replace("Z",""))
vs_files = pd.to_datetime(esgf_files["timestamp"], format="%Y%m%d").dt.tz_localize(None) # no version for files

(tss_files - vs_files).sort_values()

935      0 days 00:05:38
936      0 days 00:05:44
4847     0 days 00:09:28
4851     0 days 00:09:32
4848     0 days 00:09:35
              ...       
3297   641 days 14:15:20
3301   641 days 14:16:39
3299   641 days 14:18:08
33     936 days 14:15:41
32     936 days 14:15:50
Length: 5214, dtype: timedelta64[ns]

¿Salen los mismos numeros que las dimensiones "member" del IA-monthly?

En ia-monthly eran:

```
historical = 35
ssp126 = 32
ssp245 = 34
ssp370 = 30
ssp585 = 34
```

Esto es lo que hay en ESGF.

In [18]:
esgf["title"].str.split(".", expand=True)[[3,4,8]].groupby([3,4]).first().reset_index().groupby(4).count()[[3]]

,3
4,
historical,34
ssp126,31
ssp245,30
ssp370,28
ssp460,5
ssp585,33


Esto es lo que había en el inventario del ATLAS (`CMIP6_day.csv`).

In [19]:
instance_ids.str.split(".", expand=True)[[3,4,8]].groupby([3,4]).first().reset_index().groupby(4).count()[[3]]

,3
4,
historical,35
ssp126,32
ssp245,32
ssp370,29
ssp460,5
ssp585,34


Checksums matching? `shas` contains sha256 checksums computed from a local download as of 2025 of the files from ESGF (all of them work, only retracted missing).

In [20]:
sha = pd.read_csv("shas", header=None, names=["sha256", "file"], sep=" ")
sha

,sha256,file
0,327b97033a586dc6eb8ce6e8f345a527bffc8d594e82d9...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
1,db0636944144ae05a0b3c7878485e3fd036e598c8d669b...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
2,0f6318722dfdbcef8dd8b766c5a0c5c4f1ba6d0421e036...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
3,3daf052e0e4f9ba7d9a3b78166f60933122b0187aa08e0...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
4,2562adb7733ac0ddbe8f6b467368c0ed4bcfe3f94650df...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
...,...,...
3039,c25890d41c63ff8cb5e383d405bea02fbc2cc34fabb55e...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3040,5ce7e7ef6b50445e96fbd52fdef05a7bf1edd6f6807195...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3041,faed94a2899d0c42614d18cd4037889c1c9d6ee7c6f740...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3042,ffcbeafae2f389ab595d752e12f745983b6f574b082c61...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...


These are checksums from lustre.

In [21]:
sha_ui = pd.read_csv("shas-ui", header=None, names=["sha256", "file"], sep=" ")
sha_ui = sha_ui[sha_ui["file"].isin(sha["file"])]
sha_ui

,sha256,file
0,065a828f57e5d702e16608040cf37c57eb5f4db88e1e13...,v20191108/tas_day_NorESM2-MM_ssp245_r1i1p1f1_g...
1,927acce49d2c27a86a8da4f59d94e41ecdb9fdf4633264...,v20191108/tas_day_NorESM2-MM_ssp245_r1i1p1f1_g...
2,562403d966338b766e554768ebadea187767359c45f5bb...,v20191108/tas_day_NorESM2-MM_ssp245_r1i1p1f1_g...
3,160ce3c692b23a83ffa397cea0ed1494a058c472967d64...,v20191108/tas_day_NorESM2-MM_ssp245_r1i1p1f1_g...
4,81e645dfbef0ad3d589b8210feda0fc973d4d5d2984884...,v20191108/tas_day_NorESM2-MM_ssp245_r1i1p1f1_g...
...,...,...
3264,fe21e3fbfcda2e9c0c996a3c67095764819bbaf270048d...,v20191202/tas_day_KIOST-ESM_historical_r1i1p1f...
3265,20e949eaaaf640495f56dced938499f273b8fa0ffa10f6...,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...
3266,7cb26ece5ba2274e8878eea0a32608d751caeebdc78845...,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...
3267,f17b11cf4f55d746799a64073359741397ba30461eae33...,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...


In [22]:
sha.merge(sha_ui, on=["sha256", "file"])

,sha256,file
0,327b97033a586dc6eb8ce6e8f345a527bffc8d594e82d9...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
1,db0636944144ae05a0b3c7878485e3fd036e598c8d669b...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
2,0f6318722dfdbcef8dd8b766c5a0c5c4f1ba6d0421e036...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
3,3daf052e0e4f9ba7d9a3b78166f60933122b0187aa08e0...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
4,2562adb7733ac0ddbe8f6b467368c0ed4bcfe3f94650df...,v20180701/tas_day_GFDL-CM4_historical_r1i1p1f1...
...,...,...
3039,c25890d41c63ff8cb5e383d405bea02fbc2cc34fabb55e...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3040,5ce7e7ef6b50445e96fbd52fdef05a7bf1edd6f6807195...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3041,faed94a2899d0c42614d18cd4037889c1c9d6ee7c6f740...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...
3042,ffcbeafae2f389ab595d752e12f745983b6f574b082c61...,v20201112/tas_day_IITM-ESM_ssp126_r1i1p1f1_gn_...


¿Los checksum de esgf son iguales entre data nodes? Sí.

In [23]:
esgf_files[["title", "checksum"]].groupby(["title"]).nunique()["checksum"].unique()

array([1])

¿Coinciden con los de ESGF? Sí.

In [24]:
pd.DataFrame({
    "file": (esgf_files["dataset_id"].str.split("|").str[0].str.split(".").str[-1] + "/" + esgf_files["title"]),
    "sha256": esgf_files["checksum"],
}).drop_duplicates().merge(sha, on=["file", "sha256"])

,file,sha256
0,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...,7cb26ece5ba2274e8878eea0a32608d751caeebdc78845...
1,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...,20e949eaaaf640495f56dced938499f273b8fa0ffa10f6...
2,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...,b469e5a706009475f72c84a127a52d8cfdd22b265f4704...
3,v20191108/tas_day_ACCESS-CM2_historical_r1i1p1...,f17b11cf4f55d746799a64073359741397ba30461eae33...
4,v20191108/tas_day_ACCESS-CM2_ssp126_r1i1p1f1_g...,1ef8f0483e3609d9ff3292a6ce6bf9a769bacc59e8f713...
...,...,...
3039,v20190715/tas_day_UKESM1-0-LL_ssp245_r1i1p1f2_...,bf0defb2c5619aa151727144da8792bba20fb20d45218a...
3040,v20190726/tas_day_UKESM1-0-LL_ssp370_r1i1p1f2_...,95f183c3be024c901726a5ca9ded4f5c8fd2d69eba8b85...
3041,v20190726/tas_day_UKESM1-0-LL_ssp370_r1i1p1f2_...,12c13dbb902022cc804c6c9c53d36a1ed2a38217e65d51...
3042,v20190726/tas_day_UKESM1-0-LL_ssp585_r1i1p1f2_...,dc32687dadfce2df96e40cad1c11c6c1eab7185313bdbb...


Ok, parece que ESGF sí que tiene buena reproducibilidad después de todo. Al menos no se pierden todos los modelos que perdía con ESGF-VA (porque es OPeNDAP).

In [25]:
esgf_files["url"].apply(lambda x: len(x)).unique()

array([2, 4, 3, 1])

In [26]:
def find_opendap(urls):
    opendap_url = None
    for url in urls:
        if url.endswith("OPENDAP"):
            opendap_url = url

    return opendap_url

def find_opendap_but_dkrz(urls):
    opendap_url = find_opendap(urls)
    if opendap_url is not None and "dkrz" in opendap_url.lower():
        opendap_url = None

    return opendap_url

esgf_files["url"].apply(find_opendap).isna().sum()

26

Solo 26 files no tienen OPeNDAP endpoint pero DKRZ por ejemplo lo tiene apagado. ¿Cuántos ficheros tienen OPeNDAP y no son DKRZ?

In [27]:
len(esgf_files[~esgf_files["url"].apply(find_opendap_but_dkrz).isna()].drop_duplicates("title"))

2034

Ok, son más pero cuántos son CEDA.

In [28]:
a = esgf_files[~esgf_files["url"].apply(find_opendap_but_dkrz).isna()]
a[a["data_node"] == "esgf.ceda.ac.uk"]["dataset_id"].unique().size

105

In [29]:
b = pd.DataFrame(set([(x.split(".")[3], x.split(".")[4]) for x in a[a["data_node"] == "esgf.ceda.ac.uk"]["dataset_id"].unique()]), columns=["source_id", "experiment"])
b.groupby("experiment").count()

,source_id
experiment,
historical,23
ssp126,23
ssp245,19
ssp370,18
ssp460,3
ssp585,19


# CORDEX

In [30]:
institutes = ["AUTH-MC", "AWI", "BCCR", "BOUN", "CCCMA", "CEC", "CLMcom",
              "CLMcom-BTU", "CLMcom-CMCC", "CLMcom-DWD", "CLMcom-ETH",
              "CLMcom-HZG", "CLMcom-KIT", "CNRM", "CSIRO", "CYI", "DHMZ",
              "DMI", "DWD", "ETH", "FZJ", "FZJ-IDL", "GERICS", "HCLIMcom",
              "HMS", "ICTP", "IITM", "INPE", "IPSL", "IPSL-INERIS", "ISU",
              "KNMI", "KNU", "MGO", "MOHC", "MPI-CSC", "NCAR", "NIMS-KMA",
              "NUIM", "ORNL", "OURANOS", "PIK", "PNU", "POSTECH",
              "RMIB-UGent", "RU-CORE", "SMHI", "UA", "UCAN", "UHOH",
              "ULg", "UNIST", "UQAM"]

rcm_names = ["ALADIN52", "ALADIN53", "ALADIN63", "ALADIN64", "ALARO-0",
             "AROME41t1", "ARPEGE51", "CCAM-2008", "CCLM-0-9",
             "CCLM4-21-2", "CCLM4-8-17", "CCLM4-8-17-CLM3-5",
             "CCLM5-0-15", "CCLM5-0-2", "CCLM5-0-6", "CCLM5-0-9",
             "COSMO-crCLIM", "COSMO-crCLIM-v1-1", "CRCM5", "CRCM5-SN",
             "CanRCM4", "DeepESD-EE", "EPISODES2018", "Eta",
             "HCLIM38-AROME", "HIRHAM5", "HadGEM3-RA",
             "HadREM3-GA7-05", "HadRM3P", "ICON-2024-01", "MAR311",
             "MAR36", "RA", "RACMO21P", "RACMO22E", "RACMO22T",
             "RCA4", "RCA4-SN", "REMO2009", "REMO2015", "RRCM",
             "RegCM4", "RegCM4-0", "RegCM4-2", "RegCM4-3",
             "RegCM4-4", "RegCM4-6", "RegCM4-7", "SNURCM",
             "STARS3", "VRF370", "WETTREG2013", "WRF", "WRF331",
             "WRF331F", "WRF331G", "WRF341E", "WRF341I", "WRF351",
             "WRF361H", "WRF381BB", "WRF381BG", "WRF381BI",
             "WRF381CA", "WRF381DA", "WRF381P"]

In [31]:
ids = []
for inventory in os.listdir("cordex"):
    if not inventory.endswith(".csv"):
        print(f"Invalid inventory file {inventory}, continue...")
        continue

    df = pd.read_csv("cordex/"+inventory).drop(["ESGF_label"], axis=1)
    if "Unnamed: 0" in df.columns:
        df = df.drop("Unnamed: 0", axis=1)

    var_cols = df.columns.difference(["COPERNICUS_label"])
    out = df.melt(
            id_vars="COPERNICUS_label",
            value_vars=var_cols,
            var_name="variable",
            value_name="version"
        ).assign(
            esgf_with_var=lambda x: (
                "cordex_output_" + x["COPERNICUS_label"] + "_" + x["variable"] + "_" + x["version"]))

    
    ids.extend([x.replace("_", ".") for x in out["esgf_with_var"].dropna()])

Invalid inventory file .ipynb_checkpoints, continue...


In [32]:
ids2 = []
for i in ids:
    facets = i.split(".")
    gcm = "-".join(facets[3].split("-")[1:])
    gcm = facets[3]
    ins = facets[6].split("-")[0]
    rcm = "-".join(facets[6].split("-")[1:])
    ids2.append(".".join([
        facets[0],
        facets[1],
        facets[2],
        ins,
        gcm,
        facets[4],
        facets[5],
        rcm,
        facets[7],
        "fx" if facets[8] in ["orog", "sftlf"] else "day",
        facets[8],
        facets[9]]))

In [33]:
cordex_datasets = pd.Series(sorted(ids2), name="instance_id")
cordex_datasets = pd.DataFrame({"master_id": cordex_datasets.str.split(".").str[:-1].str.join("."), "instance_id": cordex_datasets})
cordex_datasets.to_csv("cordex.csv", index=False)
cordex_datasets

,master_id,instance_id
0,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...
1,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...
2,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...
3,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...
4,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...,cordex.output.AFR-22.CCCma.CCCma-CanESM2.histo...
...,...,...
4449,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...
4450,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...
4451,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...
4452,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...,cordex.output.WAS-44.SMHI.NOAA-GFDL-GFDL-ESM2M...
